# Details of selected Pango X lineages

Values quoted in the supplementary section "Additional analyses of selected Pango X
lineages" that are not in the Pango X table (`tab_pango_x_events.ipynb`) or the
multi-parent recombinants notebook (`analysis_multi_parent_recombinants.ipynb`, for XB).

In [1]:
import collections

import numpy as np
import pandas as pd
import tszip
import sc2ts

In [2]:
ts = tszip.load("../data/sc2ts_viridian_v2.3.trees.tsz")
df_node = sc2ts.node_data(ts, inheritance_stats=False)
df_mut = sc2ts.mutation_data(ts, inheritance_stats=False)
ds = sc2ts.Dataset("../data/viridian_mafft_2024-10-14_v1.vcz.zip", date_field="Date_tree")
sample_node = df_node[df_node["is_sample"]].reset_index().set_index("sample_id")["node_id"]

In [3]:
dfe = pd.read_csv("../data/pango_x_events.csv")


def parents(u):
    """The parent edges of node u, left to right."""
    edges = np.where(ts.edges_child == u)[0]
    edges = edges[np.argsort(ts.edges_left[edges])]
    return pd.DataFrame({
        "left": ts.edges_left[edges].astype(int),
        "right": ts.edges_right[edges].astype(int),
        "parent": ts.edges_parent[edges],
        "parent_pango": df_node.loc[ts.edges_parent[edges], "pango"].values,
        "parent_date": df_node.loc[ts.edges_parent[edges], "date"].dt.date.values,
    })


def mutations(u):
    df = df_mut[df_mut["node"] == u]
    return [f"{a}{p}{b}" for a, p, b in zip(df["inherited_state"], df["position"], df["derived_state"])]


def path_to_root(u, position):
    tree = ts.at(position)
    path = []
    while u != -1:
        path.append(u)
        u = tree.parent(u)
    return path


def describe_path(u, position, stop=None):
    rows = []
    for v in path_to_root(u, position):
        rows.append({
            "node": v,
            "pango": df_node.loc[v, "pango"],
            "sample_id": df_node.loc[v, "sample_id"],
            "date": df_node.loc[v, "date"].date(),
            "recombinant": df_node.loc[v, "is_recombinant"],
            "mutations": " ".join(mutations(v)),
        })
        if v == stop:
            break
    return pd.DataFrame(rows)


def samples_by_pango(u, position=0):
    tree = ts.at(position)
    return collections.Counter(df_node.loc[list(tree.samples(u)), "pango"])


def sample_alleles(sample_ids, positions):
    """Alleles in the Viridian alignments for the given samples and positions."""
    rows = {}
    for var in ds.variants(sample_id=sample_ids, position=positions):
        rows[var.position] = var.alleles[var.genotypes]
    return pd.DataFrame(rows, index=sample_ids)

## XM/XAL

In [4]:
dfe[dfe["pango"].isin(["XM", "XAL"])]

,root,root_pango,root_mutations,root_type,pango_samples,non_pango_samples,closest_recombinant,closest_recombinant_path_len,closest_recombinant_time,closest_recombinant_descendants,pango
18,1502293,XAL,2,I,14,{},1179001,4.0,70.676494,"{'XM': 31, 'XAL': 14, 'BA.2': 7}",XAL
24,1095100,XM,0,R,1,{'BA.1': 1},1095100,0.0,0.000000,"{'XM': 1, 'BA.1': 1}",XM
25,1179001,XM,0,R,31,"{'BA.2': 7, 'XAL': 14}",1179001,0.0,0.000000,"{'XM': 31, 'XAL': 14, 'BA.2': 7}",XM
26,1357499,XM,1,R,1,{},1357499,0.0,0.000000,{'XM': 1},XM


In [5]:
for root in dfe[dfe["pango"] == "XM"]["root"]:
    print(root, df_node.loc[root, "sample_id"])
    display(describe_path(root, 21000, stop=dfe.set_index("root").loc[root, "closest_recombinant"]))

1095100 


,node,pango,sample_id,date,recombinant,mutations
0,1095100,XM,,2021-12-04,True,


1179001 


,node,pango,sample_id,date,recombinant,mutations
0,1179001,XM,,2022-01-07,True,


1357499 


,node,pango,sample_id,date,recombinant,mutations
0,1357499,XM,,2022-02-25,True,C18131T


In [6]:
xm_recombinants = sorted(set(dfe[dfe["pango"] == "XM"]["closest_recombinant"]))
for u in xm_recombinants:
    print(u, samples_by_pango(u, 21000))
    display(parents(u))

1095100 Counter({'XM': 1, 'BA.1': 1})


,left,right,parent,parent_pango,parent_date
0,0,22599,937985,BA.1.1,2021-08-30
1,22599,29904,871741,BA.2,2021-10-19


1179001 Counter({'XM': 31, 'XAL': 14, 'BA.2': 7})


,left,right,parent,parent_pango,parent_date
0,0,21618,1090446,BA.1.1,2022-01-07
1,21618,29904,871741,BA.2,2021-10-19


1357499 Counter({'XM': 1})


,left,right,parent,parent_pango,parent_date
0,0,21762,1050102,BA.1.1,2021-12-22
1,21762,29904,1195919,BA.2,2022-02-02


The BA.2-specific 9-base deletion at 21633-21641 in the XM samples.

In [7]:
xm_samples = list(df_node[df_node["is_sample"] & (df_node["pango"] == "XM")]["sample_id"])
alleles = sample_alleles(xm_samples, list(range(21633, 21642)))
has_deletion = (alleles == "-").all(axis=1)
print(has_deletion.value_counts())
alleles[~has_deletion]

True     30
False     3
Name: count, dtype: int64


,21633,21634,21635,21636,21637,21638,21639,21640,21641
SRR19024311,T,A,C,C,C,C,C,T,G
SRR18775874,T,A,C,C,C,C,C,T,G
ERR9447529,T,A,C,C,C,C,C,T,G


In [8]:
for sample_id in alleles[~has_deletion].index:
    print(sample_id)
    display(describe_path(sample_node[sample_id], 21000, stop=1179001).head(6))

SRR19024311


,node,pango,sample_id,date,recombinant,mutations
0,1095099,XM,SRR19024311,2022-01-09,False,
1,1095100,XM,,2021-12-04,True,
2,937985,BA.1.1,,2021-08-30,False,C2470T
3,900225,BA.1.1,,2021-08-30,False,G22599A
4,871728,BA.1,,2021-08-30,False,C22202A
5,902932,BA.1,,2021-08-30,False,-22194A -22195G -22196T T22197G A22198C T22204C


SRR18775874


,node,pango,sample_id,date,recombinant,mutations
0,1357498,XM,SRR18775874,2022-03-14,False,
1,1357499,XM,,2022-02-25,True,C18131T
2,1050102,BA.1.1,SRR19321790,2021-12-22,False,C13458T
3,961203,BA.1.18,,2021-09-19,False,C21595T
4,932492,BA.1.1,,2021-09-19,False,C2470T
5,903410,BA.1.1,,2021-09-19,False,G22599A


ERR9447529


,node,pango,sample_id,date,recombinant,mutations
0,1373232,XM,ERR9447529,2022-03-19,False,G6166A T21618C -21633T -21634A -21635C -21636C...
1,1311003,XM,,2022-01-07,False,T222C
2,1179001,XM,,2022-01-07,True,


Two of the three lack the deletion because their recombination nodes switch to BA.2 to the
right of it. ERR9447529 is in the dominant clade (breakpoint to the left), and deletion
remapping reverts the deletion on its branch:

In [9]:
mutations(sample_node["ERR9447529"])

['G6166A',
 'T21618C',
 '-21633T',
 '-21634A',
 '-21635C',
 '-21636C',
 '-21637C',
 '-21638C',
 '-21639C',
 '-21640T',
 '-21641G',
 'C21762T',
 'T21765-',
 'A21766-',
 'C21767-',
 'A21768-',
 'T21769-',
 'G21770-',
 'C21846T',
 'A21987-',
 'T21988-',
 'G21989-',
 'T21990-',
 'T21991-',
 'T21992-',
 'A21993-',
 'T21994-',
 'T21995-']

## XN/XAU

In [10]:
dfe[dfe["pango"].isin(["XN", "XAU"])]

,root,root_pango,root_mutations,root_type,pango_samples,non_pango_samples,closest_recombinant,closest_recombinant_path_len,closest_recombinant_time,closest_recombinant_descendants,pango
23,1472914,XAU,1,S,9,{},-1,inf,inf,{},XAU
79,1253883,XN,1,S,141,{},-1,inf,inf,{},XN


In [11]:
for sample_id in ["ERR8986821", "ERR9502469", "ERR9453882", "ERR9823912"]:
    print(sample_id)
    display(describe_path(sample_node[sample_id], 3000).head(5))

ERR8986821


,node,pango,sample_id,date,recombinant,mutations
0,1253883,XN,ERR8986821,2022-02-13,False,G10986A
1,1394369,BA.2,,2022-01-27,False,G670T T2790C
2,1168460,BA.2,ERR11838301,2022-01-27,False,A2832G
3,932322,BA.2,,2021-10-19,False,C22792T
4,965911,BA.2.5,,2021-10-19,False,C9866T


ERR9502469


,node,pango,sample_id,date,recombinant,mutations
0,1392702,BA.2,ERR9502469,2022-03-25,False,
1,1394369,BA.2,,2022-01-27,False,G670T T2790C
2,1168460,BA.2,ERR11838301,2022-01-27,False,A2832G
3,932322,BA.2,,2021-10-19,False,C22792T
4,965911,BA.2.5,,2021-10-19,False,C9866T


ERR9453882


,node,pango,sample_id,date,recombinant,mutations
0,1353131,BA.2,ERR9453882,2022-03-13,False,
1,1353875,BA.2,,2022-01-14,False,T2790C A2832G
2,1153483,BA.2.23,,2021-12-11,False,G670T
3,965911,BA.2.5,,2021-10-19,False,C9866T
4,871741,BA.2,,2021-10-19,False,A22786C


ERR9823912


,node,pango,sample_id,date,recombinant,mutations
0,1472914,XAU,ERR9823912,2022-05-06,False,T17407C
1,1479547,BA.2,,2022-02-24,False,A24433G
2,3420435,BA.2,,2022-01-27,False,C25624T
3,1474095,BA.2,,2022-01-27,False,C2470T
4,1394369,BA.2,,2022-01-27,False,G670T T2790C


In [12]:
for root in dfe[dfe["pango"].isin(["XN", "XAU"])]["root"]:
    print(root, df_node.loc[root, "pango"])
    display(describe_path(root, 3000).head(5))

1472914 XAU


,node,pango,sample_id,date,recombinant,mutations
0,1472914,XAU,ERR9823912,2022-05-06,False,T17407C
1,1479547,BA.2,,2022-02-24,False,A24433G
2,3420435,BA.2,,2022-01-27,False,C25624T
3,1474095,BA.2,,2022-01-27,False,C2470T
4,1394369,BA.2,,2022-01-27,False,G670T T2790C


1253883 XN


,node,pango,sample_id,date,recombinant,mutations
0,1253883,XN,ERR8986821,2022-02-13,False,G10986A
1,1394369,BA.2,,2022-01-27,False,G670T T2790C
2,1168460,BA.2,ERR11838301,2022-01-27,False,A2832G
3,932322,BA.2,,2021-10-19,False,C22792T
4,965911,BA.2.5,,2021-10-19,False,C9866T


Sites used to designate XN (C241T, A2832G, C3037T) and XAU (plus C2470T).

In [13]:
sample_alleles(["ERR8986821", "ERR9502469", "ERR9453882", "ERR9823912"], [241, 2470, 2832, 3037]).join(
    df_node.set_index("sample_id")["pango"])

,241,2470,2832,3037,pango
ERR8986821,T,C,G,T,XN
ERR9502469,T,C,G,T,BA.2
ERR9453882,T,C,G,T,BA.2
ERR9823912,T,T,G,T,XAU


Independent occurrences of A2832G and C2470T within BA.2 in the ARG, with the
Pango lineages of the samples inheriting each mutation.

In [14]:
def mutation_occurrences(position, derived_state, pango_prefix):
    df = df_mut[(df_mut["position"] == position) & (df_mut["derived_state"] == derived_state)].copy()
    df["node_pango"] = df_node.loc[df["node"], "pango"].values
    df = df[df["node_pango"].str.startswith(pango_prefix)]
    df["descendants"] = [samples_by_pango(u, position) for u in df["node"]]
    return df[["node", "node_pango", "descendants"]]


df_2832 = mutation_occurrences(2832, "G", "BA.2")
print(len(df_2832), "A2832G mutations on BA.2 nodes")
df_2832

12 A2832G mutations on BA.2 nodes


,node,node_pango,descendants
386484,1168460,BA.2,"{'BA.2': 18, 'XN': 141, 'XAU': 9}"
386493,1353875,BA.2,{'BA.2': 47}
386507,1255861,BA.2.9,{'BA.2.9': 1}
386510,1476442,BA.2,{'BA.2': 2}
386511,1410046,BA.2,{'BA.2': 2}
386512,1321009,BA.2,{'BA.2': 1}
386513,1376002,BA.2.9.2,{'BA.2.9.2': 1}
386515,1445345,BA.2.9.3,{'BA.2.9.3': 1}
386516,1441267,BA.2,{'BA.2': 16}
386517,1465203,BA.2,{'BA.2': 1}


In [15]:
df_2470 = mutation_occurrences(2470, "T", "BA.2")
print(len(df_2470), "C2470T mutations on BA.2 nodes")
df_2470

29 C2470T mutations on BA.2 nodes


,node,node_pango,descendants
357694,1185016,BA.2.10,{'BA.2.10': 1}
357726,1474095,BA.2,"{'BA.2': 8, 'XAU': 9}"
357727,1190560,BA.2,{'BA.2': 1}
357728,1383613,BA.2,{'BA.2': 4}
357731,1467866,BA.2,{'BA.2': 1}
357733,1308049,BA.2,{'BA.2': 1}
357735,1352458,BA.2.8,{'BA.2.8': 1}
357740,1311224,BA.2,{'BA.2': 1}
357742,1416114,BA.2,{'BA.2': 1}
357743,1390361,BA.2.37,{'BA.2.37': 2}


## XQ and nested lineages

In [16]:
xq_lineages = ["XQ", "XR", "XU", "XAA", "XAB", "XAG", "XAM"]
dates = df_node[df_node["is_sample"] & df_node["pango"].isin(xq_lineages)]["date"]
dates.min().date(), dates.max().date()

(datetime.date(2022, 2, 12), datetime.date(2022, 8, 19))

In [17]:
dfe[dfe["pango"].isin(xq_lineages)][["root", "pango", "closest_recombinant", "closest_recombinant_path_len"]]

,root,pango,closest_recombinant,closest_recombinant_path_len
21,1361127,XR,1251110,4.0
29,1251110,XQ,1251110,0.0
32,1406901,XAA,1251110,7.0
59,1477829,XAM,1251110,7.0
65,1480519,XAB,1251110,7.0
68,1486604,XAG,1251110,7.0
80,1489452,XU,1251110,7.0


## XAC

In [18]:
df_xac = dfe[dfe["pango"] == "XAC"]
df_xac[["root", "pango_samples", "closest_recombinant", "closest_recombinant_path_len"]]

,root,pango_samples,closest_recombinant,closest_recombinant_path_len
45,1462043,8,1448928,2.0
46,1488123,9,1448928,2.0
47,1448927,1,1448928,2.0
48,1478579,1,1448928,2.0


In [19]:
u = df_xac["closest_recombinant"].iloc[0]
assert (df_xac["closest_recombinant"] == u).all()
print(u, samples_by_pango(u, 25000))
parents(u)

1448928 Counter({'XAC': 19})


,left,right,parent,parent_pango,parent_date
0,0,26028,1266436,BA.2.3,2022-01-02
1,26028,29904,998299,BA.1.1,2021-12-08


## XAJ

In [20]:
df_xaj = dfe[dfe["pango"] == "XAJ"]
df_xaj

,root,root_pango,root_mutations,root_type,pango_samples,non_pango_samples,closest_recombinant,closest_recombinant_path_len,closest_recombinant_time,closest_recombinant_descendants,pango
57,1535004,XAJ,11,I,21,{},-1,inf,inf,{},XAJ


In [21]:
xaj_root = df_xaj["root"].iloc[0]
describe_path(xaj_root, 22000).head(4)

,node,pango,sample_id,date,recombinant,mutations
0,1535004,XAJ,,2022-05-08,False,T15009C T21765- A21766- C21767- A21768- T21769...
1,1141757,BA.2.12,ERR8275142,2022-01-21,False,C11674T
2,1100773,BA.2.12,SRR18519036,2022-01-11,False,C23673T
3,963511,BA.2,SRR18518948,2021-12-01,False,C25416T


Which of the mutations on the XAJ origin node are carried by the BA.4 sample SRR18848050?

In [22]:
df = df_mut[df_mut["node"] == xaj_root][["position", "inherited_state", "derived_state"]].copy()
alleles = sample_alleles(["SRR18848050"], list(df["position"]))
df["SRR18848050"] = alleles.loc["SRR18848050", df["position"]].values
df["shared"] = df["SRR18848050"] == df["derived_state"]
df["deletion"] = df["derived_state"] == "-"
df

,position,inherited_state,derived_state,SRR18848050,shared,deletion
1279840,15009,T,C,T,False,False
1744429,21765,T,-,-,True,True
1747139,21766,A,-,-,True,True
1748406,21767,C,-,-,True,True
1749492,21768,A,-,-,True,True
1750572,21769,T,-,-,True,True
1751828,21770,G,-,-,True,True
1914391,22917,T,G,G,True,False
1924438,23018,T,G,G,True,False
1926447,23040,G,A,A,True,False


In [23]:
describe_path(sample_node["SRR18848050"], 22000).head(5)

,node,pango,sample_id,date,recombinant,mutations
0,3154509,BA.4,SRR18848050,2022-03-16,False,
1,871739,BA.4,,2022-01-07,False,A23535G
2,871740,BA.4,,2022-01-07,False,T693-
3,3423593,BA.4,,2022-01-07,False,A686- A687- G688- T689- C690- A691- T692- T694...
4,1488116,BA.4.8,,2021-11-10,False,C28724T


Genome-wide differences between the first XAJ sample, its BA.2.12 parent (ERR8275142) and
the BA.4 sample SRR18848050 in the Viridian alignments, ignoring missing and ambiguous bases.
XAJ matches BA.2.12 outside 21765-23535 and BA.4 inside it.

In [24]:
xaj_samples = df_node[df_node["is_sample"] & (df_node["pango"] == "XAJ")].sort_values("date")
xaj_sample = xaj_samples["sample_id"].iloc[0]
compare = [xaj_sample, "ERR8275142", "SRR18848050"]
alleles = sample_alleles(compare, list(range(1, 29904))).T
alleles = alleles[alleles.isin(list("ACGT-")).all(axis=1)]
for other in compare[1:]:
    diff = alleles[alleles[xaj_sample] != alleles[other]].index
    inside = (diff >= 21765) & (diff <= 23535)
    print(f"{xaj_sample} vs {other}: {len(diff)} differences, {inside.sum()} in 21765-23535")
alleles[alleles.nunique(axis=1) > 1]

ERR9858462 vs ERR8275142: 12 differences, 10 in 21765-23535
ERR9858462 vs SRR18848050: 18 differences, 0 in 21765-23535


,ERR9858462,ERR8275142,SRR18848050
686,A,A,-
687,A,A,-
688,G,G,-
689,T,T,-
690,C,C,-
691,A,A,-
692,T,T,-
693,T,T,-
694,T,T,-
1085,T,G,G


## XBB and BJ.1

In [25]:
df_meta = ds.metadata.as_dataframe(["Viridian_pangolin_1.29", "Date_tree", "Viridian_N"])
df_bj1 = df_meta[df_meta["Viridian_pangolin_1.29"] == "BJ.1"].copy()
df_bj1["in_arg"] = df_bj1.index.isin(sample_node.index)
df_bj1

,Viridian_pangolin_1.29,Date_tree,Viridian_N,in_arg
sample_id,,,,
SRR21783511,BJ.1,2022-09-02,478,False
ERR10366626,BJ.1,2022-09-27,0,False
ERR10381953,BJ.1,2022-10-05,2667,False
ERR13015043,BJ.1,2022-09-16,195,False
SRR25463698,BJ.1,.,1186,False


In [26]:
(df_node["pango"] == "BJ.1").sum()

np.int64(0)

In [27]:
xbb_root = dfe[dfe["pango"] == "XBB"]["root"].iloc[0]
print(len(mutations(xbb_root)), "mutations on the XBB origin node")
describe_path(xbb_root, 22000).head(3)

15 mutations on the XBB origin node


,node,pango,sample_id,date,recombinant,mutations
0,1673452,XBB,,2022-05-20,True,A405G C15738T T15939C T17859C A19326G T21810C ...
1,1303550,BA.2.10,ERR9201084,2022-02-26,False,G15451A
2,1207879,BA.2.10,ERR8560889,2022-02-04,False,T16342C


In [28]:
parents(dfe[dfe["pango"] == "XBB"]["closest_recombinant"].iloc[0])

,left,right,parent,parent_pango,parent_date
0,0,22577,1303550,BA.2.10,2022-02-26
1,22577,29904,1628335,BM.1.1.1,2022-04-12


## XDP/XEB

In [29]:
dfe[dfe["pango"].isin(["XDP", "XEB"])][["root", "pango", "pango_samples", "closest_recombinant",
                                        "closest_recombinant_path_len", "closest_recombinant_time"]]

,root,pango,pango_samples,closest_recombinant,closest_recombinant_path_len,closest_recombinant_time
12,1873516,XEB,5,1868729,2.0,53.352052
13,1888321,XEB,4,1874998,2.0,49.789615
77,1868729,XDP,100,1868729,0.0,0.000000
78,1874998,XDP,28,1874998,0.0,0.000000


In [30]:
xdp_recombinants = sorted(set(dfe[dfe["pango"] == "XDP"]["closest_recombinant"]))
for u in xdp_recombinants:
    print(u, df_node.loc[u, "date"].date(), samples_by_pango(u, 10000))
    display(parents(u))

1868729 2023-10-28 Counter({'XDP': 62, 'XDP.1': 24, 'XDP.3': 13, 'XEB': 5, 'JN.1.4': 3, 'XDP.2': 1})


,left,right,parent,parent_pango,parent_date
0,0,26882,1855188,JN.1.4,2023-10-28
1,26882,29904,1849244,FL.15,2023-06-12


1874998 2023-12-11 Counter({'XDP': 18, 'XDP.1': 8, 'XEB': 4, 'JN.1.4': 2, 'XDP.3': 2})


,left,right,parent,parent_pango,parent_date
0,0,27810,1869759,JN.1.4,2023-12-11
1,27810,29904,1780140,XBB.1.9.1,2022-10-06


In [31]:
for root in dfe[dfe["pango"] == "XEB"]["root"]:
    display(describe_path(root, 10000).head(4))

,node,pango,sample_id,date,recombinant,mutations
0,1873516,XEB,SRR27470671,2023-12-20,False,A405G T774C
1,1870390,XDP,,2023-10-31,False,T26882C
2,1868729,XDP,,2023-10-28,True,
3,1855188,JN.1.4,,2023-10-28,False,T18453C


,node,pango,sample_id,date,recombinant,mutations
0,1888321,XEB,,2024-01-30,False,A405G T774C
1,1881814,XDP,SRR28334834,2024-01-23,False,A11721T G25226C C25227G G25234T
2,1874998,XDP,,2023-12-11,True,
3,1869759,JN.1.4,ERR13054104,2023-12-11,False,T16260C


Both XEB origins carry A405G and T774C. Pango designates XEB with an extra FL.15
segment to the left of 774-896. Alleles of the XDP parent nodes at sites up to 896 where
they differ:

In [32]:
xdp_parents = {u: f"{df_node.loc[u, 'pango']} ({u})" for u in [1855188, 1869759, 1849244, 1780140]}
rows = {}
for var in ts.variants(samples=list(xdp_parents), isolated_as_missing=False, right=897):
    states = np.array(var.alleles)[var.genotypes]
    if len(set(states)) > 1:
        rows[int(var.site.position)] = states
pd.DataFrame(rows, index=list(xdp_parents.values()))

,405,774
JN.1.4 (1855188),A,T
JN.1.4 (1869759),A,T
FL.15 (1849244),G,C
XBB.1.9.1 (1780140),G,C


In [33]:
xeb_samples = list(df_node[df_node["is_sample"] & (df_node["pango"] == "XEB")]["sample_id"])
alleles = sample_alleles(xeb_samples, [405, 774])
alleles.value_counts()

405  774
G    C      9
Name: count, dtype: int64